In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "04-inference-engine/kv-cache")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# KV Cache from Scratch — Practice

Fill in the four blanks (**A–D**), then run the test cells below each section. Each test
targets a specific blank and prints `PASSED`, so you get immediate feedback.

**The four blanks:**
- **A** — grow the cache (concatenate new K, V onto the past)
- **B** — the causal mask
- **C** — the single decode step (feed one token + the cache)
- **D** — the cache-size formula

Everything else (the model, naive generation, the tests, plotting) is provided so you can
stay focused on the caching logic. Stuck? `01_kv_cache_worked.ipynb` is the answer key.

In [ ]:
# pip install torch matplotlib   (if needed)
import math, time, torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Attention — Blanks A and B

The cache lives here. **A**: join this step's K/V onto the remembered ones.
**B**: stop each query from seeing tokens ahead of it.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.qkv  = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        split = lambda t: t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)          # (B, n_heads, T, head_dim)

        past_len = 0
        if cache is not None:
            past_k, past_v = cache
            past_len = past_k.shape[2]
            # ---- BLANK A: grow the cache -------------------------------------
            # Glue this step's k and v onto the remembered past_k / past_v.
            # They are shaped (B, n_heads, seq, head_dim); join along the seq axis.
            k = None   # TODO: torch.cat([...], dim=?)
            v = None   # TODO: torch.cat([...], dim=?)
            # ------------------------------------------------------------------
        new_cache = (k, v)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B,H,T,past+T)

        # Absolute positions: query i is at past_len+i; key j is at j.
        q_pos = torch.arange(past_len, past_len + T, device=x.device).unsqueeze(1)
        k_pos = torch.arange(past_len + T,           device=x.device).unsqueeze(0)
        # ---- BLANK B: causal mask --------------------------------------------
        # A query must NOT see keys that lie in its future. Set those scores to
        # -inf so softmax gives them zero weight. (Which comparison of k_pos and
        # q_pos marks a key as "in the future"?)
        att = att  # TODO: att.masked_fill(<condition>, float("-inf"))
        # ----------------------------------------------------------------------
        att = att.softmax(dim=-1)

        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), new_cache

In [ ]:
class MiniLM(nn.Module):
    """Embedding -> attention layers -> output head. One cache per layer (a list)."""
    def __init__(self, vocab, d_model, n_heads, n_layers):
        super().__init__()
        self.emb    = nn.Embedding(vocab, d_model)
        self.layers = nn.ModuleList(
            CausalSelfAttention(d_model, n_heads) for _ in range(n_layers))
        self.head   = nn.Linear(d_model, vocab, bias=False)
    def forward(self, idx, caches=None):
        x = self.emb(idx)
        new_caches = []
        for i, layer in enumerate(self.layers):
            x, c = layer(x, None if caches is None else caches[i])
            new_caches.append(c)
        return self.head(x), new_caches

## Generation — Blank C

Naive is given. In cached generation, decode must feed **one** token plus the cache.

In [ ]:
@torch.no_grad()
def generate_naive(model, prompt, n_new):        # given, for reference
    idx = prompt
    for _ in range(n_new):
        logits, _ = model(idx)
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx

In [ ]:
@torch.no_grad()
def generate_cached(model, prompt, n_new):
    logits, caches = model(prompt)                    # prefill: whole prompt, no cache
    next_id = logits[:, -1].argmax(-1, keepdim=True)
    out = [prompt, next_id]
    for _ in range(n_new - 1):
        # ---- BLANK C: one decode step ------------------------------------
        # Feed ONLY the newest token plus the cache, and capture the updated
        # cache. (What are the two arguments to model here?)
        logits, caches = model(next_id)   # TODO: pass the cache too
        # ------------------------------------------------------------------
        next_id = logits[:, -1].argmax(-1, keepdim=True)
        out.append(next_id)
    return torch.cat(out, dim=1)

## Tests for A + C

Test 1 checks the full loop agrees with naive. Test 2 isolates a single decode step
against a full forward pass.

In [ ]:
# --- Test 1: caching preserves output (checks BLANK A + BLANK C) --------------
model  = MiniLM(256, 128, 4, 4).to(device).eval()
prompt = torch.randint(0, 256, (1, 16), device=device)
a = generate_naive(model,  prompt, 32)
b = generate_cached(model, prompt, 32)
assert torch.equal(a, b), "cached output != naive output -> check the cache concat / decode call"
print("Test 1 PASSED - cached matches naive:", b[0, -6:].tolist())

In [ ]:
# --- Test 2: one decode step == last position of a full forward (checks BLANK A)
m = MiniLM(256, 64, 4, 2).to(device).eval()
seq = torch.randint(0, 256, (1, 3), device=device)
full, _        = m(seq)                       # feed all 3 at once
_,  caches     = m(seq[:, :2])                # prefill first 2
step, _        = m(seq[:, 2:], caches)        # decode the 3rd using the cache
assert torch.allclose(full[:, -1], step[:, 0], atol=1e-5), "decode-with-cache disagrees with full forward"
print("Test 2 PASSED - decode step reproduces the full-sequence result")

## Test for B

Causality has a clean signature: editing a *future* token must leave *earlier* outputs
untouched. If your mask is missing, this fails.

In [ ]:
# --- Test 3: attention is causal (checks BLANK B) ----------------------------
# If the mask is right, changing a FUTURE token cannot alter an EARLIER output.
m  = MiniLM(256, 64, 4, 2).to(device).eval()
x1 = torch.randint(0, 256, (1, 6), device=device)
x2 = x1.clone(); x2[0, -1] = (x2[0, -1] + 1) % 256   # change only the last token
l1, _ = m(x1); l2, _ = m(x2)
assert torch.allclose(l1[:, :-1], l2[:, :-1], atol=1e-5), \
    "changing a future token changed the past -> causal mask is missing/wrong"
print("Test 3 PASSED - future tokens do not leak into the past")

## Memory — Blank D

The formula from the primer. Test 4 checks it against the Llama 3 8B number (128 KB/token).

In [ ]:
def kv_cache_bytes(n_layers, n_kv_heads, head_dim, seq_len, batch=1, bytes_per=2):
    # ---- BLANK D: cache size -------------------------------------------------
    # Total bytes = (K and V, so a factor of 2) x layers x kv_heads x head_dim
    #               x seq_len x batch x bytes_per_value.
    return 0   # TODO
    # --------------------------------------------------------------------------

# --- Test 4: matches the Llama 3 8B figure from the primer --------------------
per_tok = kv_cache_bytes(32, 8, 128, seq_len=1)
assert per_tok == 131072, f"expected 131072 bytes/token, got {per_tok}"
print("Test 4 PASSED - 128 KB per token")
print(f"1 user @ 128K : {kv_cache_bytes(32,8,128,128_000)/1e9:.2f} GB")
print(f"32 users @ 8K : {kv_cache_bytes(32,8,128,8192,batch=32)/1e9:.2f} GB")

## Optional: the cost curve

Once all four tests pass, run this to see the decode-phase story — naive rising with
position, cached flat.

In [ ]:
# Optional: once the tests pass, watch the decode-phase cost curve.
@torch.no_grad()
def per_step_times(model, prompt, n_new, use_cache):
    times = []
    if use_cache:
        logits, caches = model(prompt)
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        for _ in range(n_new - 1):
            t0 = time.perf_counter()
            logits, caches = model(nxt, caches)
            nxt = logits[:, -1].argmax(-1, keepdim=True)
            if device == "cuda": torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    else:
        idx = prompt
        for _ in range(n_new):
            t0 = time.perf_counter()
            logits, _ = model(idx)
            idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
            if device == "cuda": torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return times

big = MiniLM(256, 256, 8, 6).to(device).eval()
p   = torch.randint(0, 256, (1, 32), device=device)
plt.figure(figsize=(7,4))
plt.plot([t*1e3 for t in per_step_times(big,p,256,False)], label="naive")
plt.plot([t*1e3 for t in per_step_times(big,p,256,True)],  label="cached")
plt.xlabel("generation step"); plt.ylabel("ms / token"); plt.legend()
plt.title("naive grows with position; cached stays ~flat"); plt.tight_layout(); plt.show()

## Done

If all four tests pass, you have a correct KV cache: **A** and **C** make it *fast*
(linear, not quadratic), **B** keeps it *correct* (causal), and **D** is why it is the
memory bottleneck that dominates real GPU serving.